## Imports

In [1]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [2]:
import numpy as np

In [3]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    classification_report,
)

In [4]:
from ucimlrepo import fetch_ucirepo

# Heart Disease

## Data

In [5]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [6]:
heart_disease.variables

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,None,years,no
1,sex,Feature,Categorical,Sex,None,None,no
2,cp,Feature,Categorical,None,None,None,no
3,trestbps,Feature,Integer,None,resting blood pressure (on admission to the ho...,mm Hg,no
4,chol,Feature,Integer,None,serum cholestoral,mg/dl,no
5,fbs,Feature,Categorical,None,fasting blood sugar > 120 mg/dl,None,no
6,restecg,Feature,Categorical,None,None,None,no
7,thalach,Feature,Integer,None,maximum heart rate achieved,None,no
8,exang,Feature,Categorical,None,exercise induced angina,None,no
9,oldpeak,Feature,Integer,None,ST depression induced by exercise relative to ...,None,no


In [7]:
X = X.fillna(value=0)

In [8]:
y = y.copy()
y.loc[y["num"] > 0, "num"] = 1

In [9]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train)

In [10]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[92 77]


In [11]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1]
[49 42]


In [12]:
scaler = MinMaxScaler(feature_range=(0, 1))

x_train = scaler.fit_transform(x_train)

x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

In [13]:
x_train = torch.from_numpy(x_train)
x_val = torch.from_numpy(x_val)
x_test = torch.from_numpy(x_test)

y_train = torch.from_numpy(y_train.values).squeeze()
y_val = torch.from_numpy(y_val.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

## Model & Algorithms

### DataLoaders

In [14]:
batch_size = 16
shuffle = True

In [15]:
train_loader = data.DataLoader(
    data.TensorDataset(x_train, y_train), 
    batch_size = batch_size, 
    shuffle = shuffle)
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1]

In [16]:
val_loader = data.DataLoader(
    data.TensorDataset(x_val, y_val), 
    batch_size = batch_size, 
    shuffle = shuffle)
x_val = val_loader.dataset.tensors[0]
y_val = val_loader.dataset.tensors[1]

### ANFIS

In [17]:
features = heart_disease.variables["name"][:13].tolist()
features

['age',
 'sex',
 'cp',
 'trestbps',
 'chol',
 'fbs',
 'restecg',
 'thalach',
 'exang',
 'oldpeak',
 'slope',
 'ca',
 'thal']

In [18]:
model = nft.rule_reduced_ANFIS(
    input_size = x_train.shape[1],
    num_mfs = 3,
    outputs = 2,
    membership_function=nft.Gaussian_MF,
    output_type='softmax',
    features=features,
    dtype=torch.float64
)

model.init_premises(x_train)

In [19]:
print(model.get_premises_structure().to_string())

        age        sex         cp       trestbps       chol        fbs       restecg       thalach       exang       oldpeak       slope         ca       thal      
         mu sigma   mu sigma   mu sigma       mu sigma   mu sigma   mu sigma      mu sigma      mu sigma    mu sigma      mu sigma    mu sigma   mu sigma   mu sigma
rule 1  0.0  0.25  0.0  0.25  0.0  0.25      0.0  0.25  0.0  0.25  0.0  0.25     0.0  0.25     0.0  0.25   0.0  0.25     0.0  0.25   0.0  0.25  0.0  0.25  0.0  0.25
rule 2  0.5  0.25  0.5  0.25  0.5  0.25      0.5  0.25  0.5  0.25  0.5  0.25     0.5  0.25     0.5  0.25   0.5  0.25     0.5  0.25   0.5  0.25  0.5  0.25  0.5  0.25
rule 3  1.0  0.25  1.0  0.25  1.0  0.25      1.0  0.25  1.0  0.25  1.0  0.25     1.0  0.25     1.0  0.25   1.0  0.25     1.0  0.25   1.0  0.25  1.0  0.25  1.0  0.25


In [20]:
with torch.no_grad():
    pred = model(x_train)

nn.functional.cross_entropy(pred, y_train)

tensor(0.6082, dtype=torch.float64)

In [21]:
model.init_consequents(x_train, y_train, driver="gelsd", ridge_lambda=1e-3)
#model.init_consequents(x_train, y_train, driver="gelsd")

print(model.get_consequents_structure()[0].to_string(), "\n\n", model.get_consequents_structure()[1].to_string())

             age       sex        cp  trestbps      chol       fbs   restecg   thalach     exang   oldpeak     slope        ca      thal          
              c0        c1        c2        c3        c4        c5        c6        c7        c8        c9       c10       c11       c12       c13
rule 1 -0.252472 -0.317987 -0.229078  1.358854 -1.416550 -0.010671 -1.049538 -0.102785  0.004997  0.460877 -0.309254  0.131184 -1.977315  2.073660
rule 2  0.097983 -0.207684 -0.326144 -0.293075 -0.395877  0.015407 -0.038496  0.143630 -0.157738 -0.339889 -0.248145 -0.418512 -0.301235  1.503905
rule 3 -0.519575 -0.660490 -0.427670 -0.658877 -0.222500 -0.663580 -0.651651 -0.414744 -0.652123 -0.164375 -0.325781  0.016028 -0.652618 -0.651651 

              age       sex        cp  trestbps      chol       fbs   restecg   thalach     exang   oldpeak     slope        ca      thal          
              c0        c1        c2        c3        c4        c5        c6        c7        c8        c9       c1

### Learning Algorithm

In [22]:
#loss_fn = nn.functional.binary_cross_entropy
loss_fn = nn.functional.cross_entropy
epochs = 1000
optimizer = torch.optim.AdamW
params = {'lr': 0.001, "weight_decay": 0.01}
#optimizer = torch.optim.Adam
#params = {'lr': 0.0001}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=80, 
    #delta=0.0001
)

In [23]:
trainer = nft.Basic_optimizer_training_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

In [24]:
"""
trainer(model, train_loader, val_loader, verbose=True)

pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("\n ------ Evaluation ------")
print("\n")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)
"""

'\ntrainer(model, train_loader, val_loader, verbose=True)\n\npred = model.predict(x_test)\n\nacc = accuracy_score(y_test, pred)\nprec = precision_score(y_test, pred, zero_division=0)\nrecall = recall_score(y_test, pred)\nf1 = f1_score(y_test, pred, zero_division=0)\nconf_matrix = confusion_matrix(y_test, pred)\nclass_rep = classification_report(y_test, pred)\n\nprint("\n ------ Evaluation ------")\nprint("\n")\nprint("Accuracy:", acc)\nprint("Precision:", prec)\nprint("Recall:", recall)\nprint("f1 score:", f1, "\n")\n\nprint("Confusion Matrix:")\nprint(conf_matrix, "\n")\n\nprint("Classification Report:")\nprint(class_rep)\n'

### SONFIS

In [25]:
Ngrow = 20
dGrow = 0.8
Nsplit = 25
eSplit = 0.35
Nvanish = 5
lVanish = 4

max_iterations = 100

anfis_trainer = trainer

sonfis_early_stopping = nft.EarlyStopping(patience=25)
last_training_iteration = False

lse_for_new_consequents = True
lse_for_new_consequents_lambda = 1e-1

In [26]:
#sonfis = nft.alt_SONFIS(
sonfis = nft.SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    early_stopping=sonfis_early_stopping,
    lse_for_new_consequents=lse_for_new_consequents,
    lse_for_new_consequents_lambda=lse_for_new_consequents_lambda,
    last_training_iteration=last_training_iteration
)

In [27]:
%time sonfis(model, train_loader, val_loader, verbose=True)

ITERATION:   0/100



STARTING STATE:
   premises                                                                                                                                                                                                                                                           output 1 consequents                                                                                                                                   output 2 consequents                                                                                                                                  
        age                 sex                  cp            trestbps                chol                 fbs             restecg             thalach               exang             oldpeak               slope                  ca                thal                            age       sex        cp  trestbps      chol       fbs   restecg   thalach     exang   oldpeak     slope        ca      thal                 

## Evaluation

### Test data

In [28]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, pred)
#mul_label_conf_matrix = multilabel_confusion_matrix(y_test, pred)
class_rep = classification_report(y_test, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

#print("Multilabel Confusion Matrix:")
#print(mul_label_conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.8461538461538461
Precision: 0.8333333333333334
Recall: 0.8333333333333334
f1 score: 0.8333333333333334 

Confusion Matrix:
[[42  7]
 [ 7 35]] 

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86        49
           1       0.83      0.83      0.83        42

    accuracy                           0.85        91
   macro avg       0.85      0.85      0.85        91
weighted avg       0.85      0.85      0.85        91



### Training data

In [29]:
pred = model.predict(x_train)

acc = accuracy_score(y_train, pred)
prec = precision_score(y_train, pred, zero_division=0)
recall = recall_score(y_train, pred)
f1 = f1_score(y_train, pred, zero_division=0)
conf_matrix = confusion_matrix(y_train, pred)
class_rep = classification_report(y_train, pred)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", recall)
print("f1 score:", f1, "\n")

print("Confusion Matrix:")
print(conf_matrix, "\n")

print("Classification Report:")
print(class_rep)

Accuracy: 0.8994082840236687
Precision: 0.9054054054054054
Recall: 0.8701298701298701
f1 score: 0.8874172185430463 

Confusion Matrix:
[[85  7]
 [10 67]] 

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.92      0.91        92
           1       0.91      0.87      0.89        77

    accuracy                           0.90       169
   macro avg       0.90      0.90      0.90       169
weighted avg       0.90      0.90      0.90       169

